# Inference speed
Kies de **act**-kernel. Download checkpoints met `scripts/fetch_checkpoints.py`, voer daarna de cellen uit.
Zet een model op `None` om het over te slaan.

Batch 1, vooraf geladen input, warm-up en CUDA-synchronisatie. Exclusief laden, beeldverwerking,
pointcloud-opbouw/FPS en transfers. Calls/s telt actiechunks, geen robotbesturingsfrequentie.

`steps` = samplerstappen; `unet_evaluations` = netwerkberekeningen per actiechunk.
DP 16 DDIM en FM 8 Heun kosten beide 16 evaluaties, zonder garantie op gelijke kwaliteit.
Test minder FM-stappen door `FM_STEPS` te wijzigen. DP3 gebruikt standaard 10.

In [ ]:
from pathlib import Path

# Normaal start Jupyter in de map van dit notebook. Zo nodig: vul REPO zelf in.
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts/benchmark_inference.py").is_file()), None)
if REPO is None:
    raise RuntimeError("Stel REPO in op de lokale greenhouse-scara-map.")

DATASET = REPO.parent / "datasets/greenhouse_dummy_dataset"
CHECKPOINTS = {
    "act": REPO / "checkpoints/act/policy_last.ckpt",
    "dp": REPO / "checkpoints/dp/latest.ckpt",
    "fm": REPO / "checkpoints/fm/latest.ckpt",
    "dp3": REPO / "checkpoints/dp3/latest.ckpt",
    "fm3": None,  # REPO / "checkpoints/fm3/latest.ckpt" na de smoketest
}
DEVICE = "auto"  # "cuda" om GPU verplicht te maken, of "cpu"
EPISODE = 0
STATES = 3
WARMUP = 5
ITERATIONS = 50
CPU_THREADS = 1
DP_STEPS = 16  # DDIM: 16 U-Net-evaluaties; training had 100 tijdstippen
FM_STEPS = 8  # Heun: 8 stappen = 16 U-Net-evaluaties
FM_METHOD = "heun"  # "euler": één evaluatie per stap
DP3_STEPS = 10  # Standaard DP3; zet op 16 voor hetzelfde aantal evaluaties als DP
FM3_STEPS = 8
FM3_METHOD = "heun"

In [ ]:
import gc
import json
import sys
from types import SimpleNamespace
import pandas as pd
import torch
from IPython.display import display

sys.path.insert(0, str(REPO / "scripts"))
from benchmark_inference import benchmark

for name, value in {"STATES": STATES, "WARMUP": WARMUP, "ITERATIONS": ITERATIONS,
                    "CPU_THREADS": CPU_THREADS, "DP_STEPS": DP_STEPS, "FM_STEPS": FM_STEPS, "DP3_STEPS": DP3_STEPS, "FM3_STEPS": FM3_STEPS}.items():
    if value is not None and value < 1:
        raise ValueError(f"{name} moet positief zijn")

device = torch.device(("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE)
if device.type not in ("cpu", "cuda"):
    raise ValueError("Kies cpu, cuda of auto")
if device.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is niet beschikbaar in deze notebookkernel")
torch.set_num_threads(CPU_THREADS)
hardware = torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU"
print(f"Kernel: {sys.executable}\nDevice: {device} ({hardware})\nPyTorch: {torch.__version__}")

In [ ]:
# Modellen worden één voor één geladen; laden telt niet mee in de timing.
episode = Path(DATASET).expanduser() / f"episode_{EPISODE}.hdf5"
selected = {k: Path(v).expanduser() for k, v in CHECKPOINTS.items() if v is not None}
if not selected:
    raise ValueError("Selecteer minimaal één checkpoint")
required = [episode, *selected.values()]
if "act" in selected:
    required += [selected["act"].parent / name for name in ("config.pkl", "dataset_stats.pkl")]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Ontbrekende bestanden:\n" + "\n".join(missing))

args = SimpleNamespace(states=STATES, warmup=WARMUP, iterations=ITERATIONS,
                       dp_steps=DP_STEPS, fm_steps=FM_STEPS, dp3_steps=DP3_STEPS, fm_method=FM_METHOD,
                       fm3_steps=FM3_STEPS, fm3_method=FM3_METHOD)
results = []
for model, checkpoint in selected.items():
    torch.manual_seed(42)
    results.append(benchmark(model, checkpoint.resolve(), episode.resolve(), args, device))
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

columns = ["model", "mean_ms", "median_ms", "p95_ms", "calls_per_second",
           "history", "sampler", "steps", "unet_evaluations"]
display(pd.DataFrame(results)[columns].round(2))

Optioneel: sla de resultaten hieronder op als CSV en JSON. Vergelijk ook het aantal samplerstappen; een smoketest meet geen policykwaliteit.

In [ ]:
output = REPO / "runs/inference_speed"
output.mkdir(parents=True, exist_ok=True)
# Elke meting krijgt een eigen bestandsnaam.
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
report = {"hardware": hardware, "torch": str(torch.__version__), "cpu_threads": CPU_THREADS,
          "episode": str(episode.resolve()), "results": results}
(output / f"{stamp}.json").write_text(json.dumps(report, indent=2) + "\n")
pd.DataFrame(results).to_csv(output / f"{stamp}.csv", index=False)
print(f"Opgeslagen: {output / stamp} (.json en .csv)")